# Merge TRIPOD References

This notebook merges the reference files from the different instances of TRIPOD and TRIPOD+AI in PubMed

In [1]:
import pandas as pd
import glob
import os

REFERENCES_FOLDER = "../data/00_references"
TRIPOD_FOLDER = f"{REFERENCES_FOLDER}/TRIPOD"
TRIPOD_AI_FOLDER = f"{REFERENCES_FOLDER}/TRIPOD_AI"

# Papers referencing TRIPOD Statement

This gathers all of the records citing any of the TRIPOD Statement entries.

In [2]:
tripod_csv_files = glob.glob(os.path.join(TRIPOD_FOLDER, "*_11_Aug_25.csv"))

In [3]:
dfs_TRIPOD = {}

for file in tripod_csv_files:
    # Extract the integer prefix from filename
    base = os.path.basename(file)
    int_name = int(base.split("_")[0])  # Convert to int
    
    # Read CSV and remove the first row
    df = pd.read_csv(file)
    df = df.iloc[1:].reset_index(drop=True)
    
    # Store in dict
    dfs_TRIPOD[int_name] = df

In [4]:
len(dfs_TRIPOD.keys()) # there are 12 entries of TRIPOD in Pubmed

12

In [5]:
# Step 1: Concatenate all dataframes into one
TRIPOD_combined_df = pd.concat(dfs_TRIPOD.values(), ignore_index=True)
print(f"Initial combined number of references: {TRIPOD_combined_df.shape[0]}")

# Step 2: Remove duplicates based on PMID
before = TRIPOD_combined_df.shape[0]
TRIPOD_combined_df = TRIPOD_combined_df.drop_duplicates(subset=["PMID"], keep="first")
after = TRIPOD_combined_df.shape[0]
print(f"Removed {before - after} duplicate PMIDs. New number of references: {TRIPOD_combined_df.shape[0]}")

# Step 3: Remove rows where PMID equals one of the TRIPOD_PMIDs
TRIPOD_PMIDs = set(dfs_TRIPOD.keys())  # the integer prefixes from file names
before = TRIPOD_combined_df.shape[0]
mask = TRIPOD_combined_df["PMID"].isin(TRIPOD_PMIDs)
rows_to_drop = TRIPOD_combined_df[mask]
print(f"Removed {len(rows_to_drop)} rows that were one of the TRIPOD statements.")
TRIPOD_combined_df = TRIPOD_combined_df[~mask]
after = TRIPOD_combined_df.shape[0]
print(f"Final number of references: {TRIPOD_combined_df.shape[0]}")

# Reset index for cleanliness
TRIPOD_combined_df = TRIPOD_combined_df.reset_index(drop=True)

Initial combined number of references: 6762
Removed 453 duplicate PMIDs. New number of references: 6309
Removed 3 rows that were one of the TRIPOD statements.
Final number of references: 6306


In [6]:
TRIPOD_combined_df.to_csv(f"{TRIPOD_FOLDER}/TRIPOD_combined_references.csv", index=False)

# Papers referencing TRIPOD+AI Statement

This gathers all of the records citing any of the TRIPOD+AI Statement entries.

In [7]:
tripod_ai_csv_files = glob.glob(os.path.join(TRIPOD_AI_FOLDER, "*_11_Aug_25.csv"))

In [8]:
dfs_TRIPOD_AI = {}

for file in tripod_ai_csv_files:
    # Extract the integer prefix from filename
    base = os.path.basename(file)
    int_name = int(base.split("_")[0])  # Convert to int
    
    # Read CSV and remove the first row
    df = pd.read_csv(file)
    df = df.iloc[1:].reset_index(drop=True)
    
    # Store in dict
    dfs_TRIPOD_AI[int_name] = df

In [9]:
len(dfs_TRIPOD_AI.keys()) # there are 2 entries of TRIPOD+AI in Pubmed

2

In [10]:
# Step 1: Concatenate all dataframes into one
TRIPOD_AI_combined_df = pd.concat(dfs_TRIPOD_AI.values(), ignore_index=True)
print(f"Initial combined number of references: {TRIPOD_AI_combined_df.shape[0]}")

# Step 2: Remove duplicates based on PMID
before = TRIPOD_AI_combined_df.shape[0]
TRIPOD_AI_combined_df = TRIPOD_AI_combined_df.drop_duplicates(subset=["PMID"], keep="first")
after = TRIPOD_AI_combined_df.shape[0]
print(f"Removed {before - after} duplicate PMIDs. New number of references: {TRIPOD_AI_combined_df.shape[0]}")

# Step 3: Remove rows where PMID equals one of the TRIPOD_AI_PMIDs
TRIPOD_AI_PMIDs = set(dfs_TRIPOD_AI.keys())  # the integer prefixes from file names
before = TRIPOD_AI_combined_df.shape[0]
mask = TRIPOD_AI_combined_df["PMID"].isin(TRIPOD_AI_PMIDs)
rows_to_drop = TRIPOD_AI_combined_df[mask]
print(f"Removed {len(rows_to_drop)} rows that were one of the TRIPOD+AI statements.")
TRIPOD_AI_combined_df = TRIPOD_AI_combined_df[~mask]
after = TRIPOD_AI_combined_df.shape[0]
print(f"Final number of references: {TRIPOD_AI_combined_df.shape[0]}")

# Reset index for cleanliness
TRIPOD_AI_combined_df = TRIPOD_AI_combined_df.reset_index(drop=True)

Initial combined number of references: 411
Removed 1 duplicate PMIDs. New number of references: 410
Removed 0 rows that were one of the TRIPOD+AI statements.
Final number of references: 410


In [11]:
TRIPOD_AI_combined_df.to_csv(f"{TRIPOD_AI_FOLDER}/TRIPOD_AI_combined_references.csv", index=False)

# Merge TRIPOD and TRIPOD+AI references

In [12]:
df_combined = pd.concat([TRIPOD_combined_df, TRIPOD_AI_combined_df], ignore_index=True)

In [13]:
exclude_PMIDs = TRIPOD_PMIDs.union(TRIPOD_AI_PMIDs)

In [14]:
df_TRIPOD = pd.read_csv(f"{REFERENCES_FOLDER}/TRIPOD/TRIPOD_combined_references.csv")
df_TRIPOD_AI = pd.read_csv(f"{REFERENCES_FOLDER}/TRIPOD_AI/TRIPOD_AI_combined_references.csv")

In [15]:
df_combined = pd.concat([df_TRIPOD, df_TRIPOD_AI], ignore_index=True)

In [16]:
# Some papers appear twice if they cite both TRIPOD and TRIPOD+AI, so we remove them
# here. They are only removed if they are exact duplicates across all columns.

df_combined_dedup = df_combined.drop_duplicates().reset_index(drop=True)

In [17]:
# Checking for uniqueness to ensure there were no rows sharing the same PMID but with different data

assert df_combined_dedup["PMID"].is_unique, "PMID is not unique"

In [18]:
df_combined_dedup.to_parquet(f"{REFERENCES_FOLDER}/merged_references.parquet.br", index=False, compression="brotli")

In [19]:
df_combined_dedup

,PMID,Title,Authors,Citation,First Author,Journal/Book,Publication Year,Create Date,PMCID,NIHMS ID,DOI
0,35799302,Development and validation of a multivariable ...,"Iachkine J, Buetti N, de Grooth HJ, Briant AR,...",Crit Care. 2022 Jul 7;26(1):205. doi: 10.1186/...,Iachkine J,Crit Care,2022,2022/07/07,PMC9261073,NaN,10.1186/s13054-022-04078-x
1,40151494,Development and validation of a risk-predictio...,"Qian J, Ruan C, Cai Y, Yi W, Liu J, Xu R.",Ther Adv Drug Saf. 2025 Mar 22;16:204209862513...,Qian J,Ther Adv Drug Saf,2025,2025/03/28,PMC11946284,NaN,10.1177/20420986251328687
2,27189013,Opportunities and challenges in developing ris...,"Goldstein BA, Navar AM, Pencina MJ, Ioannidis JP.",J Am Med Inform Assoc. 2017 Jan;24(1):198-208....,Goldstein BA,J Am Med Inform Assoc,2017,2016/05/19,PMC5201180,NaN,10.1093/jamia/ocw042
3,26862030,Prognostic factors for cerebral palsy and moto...,"Linsell L, Malouf R, Morris J, Kurinczuk JJ, M...",Dev Med Child Neurol. 2016 Jun;58(6):554-69. d...,Linsell L,Dev Med Child Neurol,2016,2016/02/11,PMC5321605,EMS67400,10.1111/dmcn.12972
4,36383295,28-day sepsis mortality prediction model from ...,"Xie Y, Zhuang D, Chen H, Zou S, Chen W, Chen Y.",Eur J Clin Microbiol Infect Dis. 2023 Jan;42(1...,Xie Y,Eur J Clin Microbiol Infect Dis,2023,2022/11/16,PMC9816294,NaN,10.1007/s10096-022-04517-1
...,...,...,...,...,...,...,...,...,...,...,...
6653,39885243,A comprehensive evaluation of histopathology f...,"Breen J, Allen K, Zucker K, Godson L, Orsi NM,...",NPJ Precis Oncol. 2025 Jan 30;9(1):33. doi: 10...,Breen J,NPJ Precis Oncol,2025,2025/01/30,PMC11782474,NaN,10.1038/s41698-025-00799-8
6654,40139886,Ensemble Deep Learning Algorithm for Structura...,"Dhingra LS, Aminorroaya A, Sangha V, Pedroso A...",J Am Coll Cardiol. 2025 Apr 1;85(12):1302-1313...,Dhingra LS,J Am Coll Cardiol,2025,2025/03/26,PMC12199746,NIHMS2088734,10.1016/j.jacc.2025.01.030
6655,39417095,An Ensemble Deep Learning Algorithm for Struct...,"Dhingra LS, Aminorroaya A, Sangha V, Pedroso A...",medRxiv [Preprint]. 2024 Dec 27:2024.10.06.243...,Dhingra LS,medRxiv,2024,2024/10/17,PMC11483021,NaN,10.1101/2024.10.06.24314939
6656,39529875,Detection of breast cancer using machine learn...,"Harnischmacher N, Rodner E, Schmitz CH.",J Biomed Opt. 2024 Nov;29(11):115001. doi: 10....,Harnischmacher N,J Biomed Opt,2024,2024/11/12,PMC11552526,NaN,10.1117/1.JBO.29.11.115001
